In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from scipy.stats import t
import matplotlib.pyplot as plt
import matplotlib
from sklearn.pipeline import Pipeline

from statsmodels.stats.sandwich_covariance import cov_hac #heteroscedasticity and autocorrelation robust covariance matrix (Newey-West)
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm

In [ ]:
df_final_uk = pd.read_excel("data/processed/df_final_uk.xlsx")

In [ ]:
# Convertir la colonne Date en Period mensuel
df_final_uk["Date"] = pd.to_datetime(df_final_uk["Date"], dayfirst=True, errors="coerce").dt.to_period("M")

# Ensuite, filtrer
df_final_uk = df_final_uk[df_final_uk["Date"] >= pd.Period("1995-07", freq="M")].reset_index(drop=True)
df_final_uk = df_final_uk.sort_values("Date").reset_index(drop=True) 

In [ ]:
nan_percent = df_final_uk.isna().mean().sort_values(ascending=False) * 100
print(nan_percent)

In [ ]:
df_final_uk = df_final_uk.drop(columns=["rd_mve"])

In [ ]:
df_final_uk = df_final_uk[~df_final_uk["Ticker"].isin(["SPX", "WEIR"])] #après normalisation : était encore vide donc trop de NaN on préfère les enlever
df_final_uk.reset_index(drop=True, inplace=True)

In [ ]:
n_unique_tickers = df_final_uk["Ticker"].nunique()
print(f"Nombre de tickers uniques : {n_unique_tickers}")

In [ ]:
covariates = ["dolvol", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "sp"]

In [ ]:
print(len(covariates))

In [ ]:
df_final = df_final_uk.copy()

In [ ]:
#check duplicate + columns
dup = df_final.duplicated(["Date","Ticker"]).sum()

print("Dupli init (Date,Ticker):", dup)
print(df_final.columns)

In [ ]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=192):
    """
    Génère des splits temporels:
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 306 mois (= 85% de la data)
    - Validation = fenêtre fixe glissante de 1 an
    - Test = 1 an 
    - Avance de step_months à chaque itération : 1 an

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop quand on a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Décale fenêtre de un → on réactualise tous les 1 ans
        start += step_months

    return splits

In [ ]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = X_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [ ]:
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target])
    y = subset[target]
    return x, y

def get_y_ha(df, idx, col="ha_global"):
    subset = df.loc[idx].copy()
    return subset[col]

In [ ]:
#BENCHMARK HA : déjà créé avant on l'exporte et on le coupe "comme il faut"
df_ha = pd.read_excel("data/processed/df_ha_uk.xlsx")
df_ha = df_ha[df_ha["Date"] >= "1995-07"].reset_index(drop=True)
df_ha = df_ha[df_ha["Date"] < "2020-12"].reset_index(drop=True)
df_ha = df_ha[~df_ha["Ticker"].isin(["SPX", "WEIR"])]


In [ ]:
#robustesse + check 

for name, df in [("df_final", df_final), ("df_ha", df_ha)]:
    print(f"\n--- {name} ---")
    print("Première date :", df["Date"].min())
    print("Dernière date :", df["Date"].max())
    print("Nombre de lignes :", len(df))
    print("Doublons (Date,Ticker) :", df.duplicated(["Date","Ticker"]).sum())

In [ ]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

In [ ]:
"""Fonctions pour nos métriques : 
def r2: mesure le r2 selon la définition de Gu et al 
% ratio : success ratio, semblable à ce qui est fait dans le papier de Xiu et Liu
R2 benchmark : on compare le R2 de nos modèles à l'historical average 
"""

#r2
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

In [ ]:
print(df_final_uk.dtypes)

In [ ]:
#OLS

r2_in_ols, r2_oos_ols = [], []
success_ratio_in_ols, success_ratio_oos_ols = [], []
df_in_ols, df_oos_ols = [], []
feature_importance_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    y_trainval_pred = ols.predict(x_trainval)
    y_test_pred = ols.predict(x_test[covariates])

    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_ols.append(r2_in)
    r2_oos_ols.append(r2_out)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    feature_importance_ols.append(np.abs(ols.coef_))

    df_in_split = pd.DataFrame({
        "Split":  split_idx,
        "Date":   dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_ols": y_trainval_pred
    })
    df_in_ols.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split":  split_idx,
        "Date":   x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_ols": y_test_pred
    })
    df_oos_ols.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_ols = pd.concat(df_in_ols, ignore_index=True)
df_in_ols = df_in_ols.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")

df_oos_ols = pd.concat(df_oos_ols, ignore_index=True)

#Checks
print(f"Moyenne R² IN  : {np.mean(r2_in_ols):.6f}")
print(f"Moyenne R² OOS : {np.mean(r2_oos_ols):.6f}")

print("In sample lines:",  df_in_ols.shape[0],  "| Duplicates (Date,Ticker) =", df_in_ols.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_ols.shape[0], "| Duplicates (Date,Ticker) =", df_oos_ols.duplicated(["Date","Ticker"]).sum())

In [ ]:
#PLS 
r2_in_pls, r2_oos_pls = [], []
success_ratio_in_pls, success_ratio_oos_pls = [], []
df_in_pls, df_oos_pls = [], []
best_components_pls, mse_val_grids = [], []

max_k = len(covariates)

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    # Sélection du meilleur k
    candidate_ks = range(1, max_k)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)
        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    best_components_pls.append(best_k)
    mse_val_grids.append(mse_val_grid)

    # Réentraînement final
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()

    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_pls.append(r2_in)
    r2_oos_pls.append(r2_out)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_pls": y_trainval_pred,
        "Split": split_idx
    })
    df_in_pls.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_pls": y_test_pred
    })
    df_oos_pls.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_pls = pd.concat(df_in_pls, ignore_index=True)
df_in_pls = df_in_pls.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")

df_oos_pls = pd.concat(df_oos_pls, ignore_index=True)

#Checks 
print("In sample lines:",  df_in_pls.shape[0],  "| Duplicates (Date,Ticker) =", df_in_pls.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_pls.shape[0], "| Duplicates (Date,Ticker) =", df_oos_pls.duplicated(["Date","Ticker"]).sum())

In [ ]:
#PCR
r2_in_pcr, r2_oos_pcr = [], []
success_ratio_in_pcr, success_ratio_oos_pcr = [], []
feature_importance_pcr = []
best_components_pcr, mse_val_grids_pcr = [], []
df_in_pcr, df_oos_pcr = [], []

max_k = len(covariates)
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    # Tuning
    candidate_ks = range(1, max_k)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)
        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)

    # Final fit
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # Importance des variables
    reg_coef = pcr_final.named_steps['reg'].coef_  # (k,)
    pca_components = pcr_final.named_steps['pca'].components_  # (k, n_features)
    projected_coefs = np.abs(reg_coef @ pca_components)  # (n_features,)
    feature_importance_pcr.append(projected_coefs.flatten())

    # Prédictions
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_pcr.append(r2_in)
    r2_oos_pcr.append(r2_out)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_pcr": y_trainval_pred,
        "Split": split_idx
    })
    df_in_pcr.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_pcr": y_test_pred
    })
    df_oos_pcr.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_pcr = pd.concat(df_in_pcr, ignore_index=True)
df_in_pcr = df_in_pcr.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_pcr = pd.concat(df_oos_pcr, ignore_index=True)

#Checks 
print("In sample lines:",  df_in_pcr.shape[0],  "| Duplicates (Date,Ticker) =", df_in_pcr.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_pcr.shape[0], "| Duplicates (Date,Ticker) =", df_oos_pcr.duplicated(["Date","Ticker"]).sum())

In [ ]:
#ENET
# ElasticNet : Elastic Net
# Hyperparamètres :
# - lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
# - l1_ratio fixé à 0.5

r2_in_en, r2_oos_en = [], []
success_ratio_in_en, success_ratio_oos_en = [], []
y_trainval_en = []

# Hyperparamètres spécifiques
best_lambdas = []
nonzero_counts_en = []
feature_importance_en = []

df_in_en, df_oos_en = [], []

enet_param_grid = {
    'alpha': np.logspace(-4, 0, num=10)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha (lambda)
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Entraînement final sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    y_trainval_pred = en_final.predict(x_trainval)
    y_test_pred = en_final.predict(x_test[covariates])

    # Importance et sparsité
    coefs = np.abs(en_final.coef_)
    feature_importance_en.append(coefs)
    nonzero_count = np.sum(en_final.coef_ != 0)
    nonzero_counts_en.append(nonzero_count)

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_en.append(r2_in)
    r2_oos_en.append(r2_out)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)
    y_trainval_en.append(y_trainval_pred)

    # Dataframes
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_en": y_trainval_pred,
        "Split": split_idx
    })
    df_in_en.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_en": y_test_pred
    })
    df_oos_en.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_en = pd.concat(df_in_en, ignore_index=True)
df_in_en = df_in_en.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_en = pd.concat(df_oos_en, ignore_index=True)

#Checks 
print("In sample lines:",  df_in_en.shape[0],  "| Duplicates (Date,Ticker) =", df_in_en.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_en.shape[0], "| Duplicates (Date,Ticker) =", df_oos_en.duplicated(["Date","Ticker"]).sum())


In [ ]:
#RF
param_grid_rf = {
    'n_estimators': [200, 400],       
    'max_depth': [3, 4, 5],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['log2', 1, 2]
}

r2_in_rf, r2_oos_rf = [], []
success_ratio_in_rf, success_ratio_oos_rf = [], []
feature_importance_rf = []
df_in_rf, df_oos_rf = [], []
best_params_rf, mse_val_grids_rf = [], []
y_trainval_rf = []

# Grid Search + Entraînement
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(**params, n_jobs=-1, random_state=0)
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Entraînement final
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    rf_final = RandomForestRegressor(**best_params, n_jobs=-1, random_state=0)
    rf_final.fit(x_trainval, y_trainval)

    y_trainval_pred = rf_final.predict(x_trainval)
    y_test_pred = rf_final.predict(x_test[covariates])

    # R² & Success Ratio
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_rf.append(r2_in)
    r2_oos_rf.append(r2_out)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)
    feature_importance_rf.append(rf_final.feature_importances_)
    y_trainval_rf.append(y_trainval_pred)

    # DataFrames
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_rf": y_trainval_pred,
        "Split": split_idx
    })
    df_in_rf.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_rf": y_test_pred
    })
    df_oos_rf.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_rf = pd.concat(df_in_rf, ignore_index=True)
df_in_rf = df_in_rf.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_rf = pd.concat(df_oos_rf, ignore_index=True)

#Checks 
print("In sample lines:",  df_in_rf.shape[0],  "| Duplicates (Date,Ticker) =", df_in_rf.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_rf.shape[0], "| Duplicates (Date,Ticker) =", df_oos_rf.duplicated(["Date","Ticker"]).sum())

In [ ]:
#GBRT
from sklearn.ensemble import GradientBoostingRegressor

r2_in_gbrt, r2_oos_gbrt = [], []
success_ratio_in_gbrt, success_ratio_oos_gbrt = [], []
feature_importance_gbrt, complexity_gbrt = [], []
df_in_gbrt, df_oos_gbrt = [], []
y_trainval_gbrt = []

best_params_gbrt = []
mse_val_grids_gbrt = []

param_grid_gbrt = {
    'n_estimators': [300],
    'learning_rate': [0.01],
    'max_depth': [2, 3, 4],
    'loss': ['huber'],
    'alpha': [0.9]
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid_gbrt):
        model = GradientBoostingRegressor(**params, random_state=0)
        model.fit(x_train[covariates], y_train)
        y_val_pred = model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"Split {split_idx} — meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train final
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    y_trainval_pred = gbrt_final.predict(x_trainval)
    y_test_pred = gbrt_final.predict(x_test[covariates])

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_gbrt.append(r2_in)
    r2_oos_gbrt.append(r2_out)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Feature importance
    importances = gbrt_final.feature_importances_
    feature_importance_gbrt.append(importances)
    complexity_gbrt.append(np.sum(importances > 0))

    # DataFrames
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_gbrt": y_trainval_pred,
        "Split": split_idx
    })
    df_in_gbrt.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_gbrt": y_test_pred
    })
    df_oos_gbrt.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat
df_in_gbrt = pd.concat(df_in_gbrt, ignore_index=True)
df_in_gbrt = df_in_gbrt.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_gbrt = pd.concat(df_oos_gbrt, ignore_index=True)

#Checks 
print("In sample lines:",  df_in_gbrt.shape[0],  "| Duplicates (Date,Ticker) =", df_in_gbrt.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_gbrt.shape[0], "| Duplicates (Date,Ticker) =", df_oos_gbrt.duplicated(["Date","Ticker"]).sum())

In [ ]:
#XGBOOST
# Initialisations
r2_in_xgb, r2_oos_xgb = [], []
success_ratio_in_xgb, success_ratio_oos_xgb = [], []
feature_importance_xgb = []
df_in_xgb, df_oos_xgb = [], []
y_trainval_xgb = []

# Grille de recherche
param_grid = {
    'n_estimators': [200, 400],       
    'max_depth': [2, 3],
    'learning_rate': [0.01]
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    # Recherche du meilleur modèle (validation)
    best_mse = float('inf')
    best_params = None

    for params in ParameterGrid(param_grid):
        model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        model.fit(x_train[covariates], y_train)
        preds = model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, preds)

        if mse < best_mse:
            best_mse = mse
            best_params = params

    print(f"Split {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Réentraînement sur train + val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    model = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    model.fit(x_trainval, y_trainval)

    y_trainval_pred = model.predict(x_trainval)
    y_test_pred = model.predict(x_test[covariates])

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_xgb.append(r2_in)
    r2_oos_xgb.append(r2_out)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Importance des variables
    importances = model.feature_importances_
    feature_importance_xgb.append(importances)

    # Prédictions in-sample
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_xgb": y_trainval_pred,
        "Split": split_idx
    })
    df_in_xgb.append(df_in_split)

    # Prédictions OOS
    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_xgb": y_test_pred
    })
    df_oos_xgb.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_xgb = pd.concat(df_in_xgb, ignore_index=True)
df_in_xgb = df_in_xgb.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_xgb = pd.concat(df_oos_xgb, ignore_index=True)

#Checks 
print("In sample lines:",  df_in_xgb.shape[0],  "| Duplicates (Date,Ticker) =", df_in_xgb.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_xgb.shape[0], "| Duplicates (Date,Ticker) =", df_oos_xgb.duplicated(["Date","Ticker"]).sum())

III. Results metrics (R2, Success ratios)

In [ ]:
#Découpage de HA

df_in_ha = df_ha[(df_ha["Date"] >= "1995-07") & (df_ha["Date"] <= "2019-11")].copy()
df_oos_ha = df_ha[(df_ha["Date"] >= "2007-12") & (df_ha["Date"] <= "2020-11")].copy()
df_in_ha["Date"]  = pd.to_datetime(df_in_ha["Date"]).dt.to_period("M")
df_oos_ha["Date"] = pd.to_datetime(df_oos_ha["Date"]).dt.to_period("M")

#Vérif
print("HA In-sample :", df_in_ha["Date"].min(), "→", df_in_ha["Date"].max())
print("HA OOS       :", df_oos_ha["Date"].min(), "→", df_oos_ha["Date"].max())
print("In-sample OLS :", df_in_ols["Date"].min(), "→", df_in_ols["Date"].max())
print("OOS OLS      :", df_oos_ols["Date"].min(), "→", df_oos_ols["Date"].max())
print("df_in_ha   :", len(df_in_ha))
print("df_oos_ha  :", len(df_oos_ha))
print("df_in_ols  :", len(df_in_ols))
print("df_oos_ols :", len(df_oos_ols))

In [ ]:
#R² - Gu et al. 

models = {
    "OLS":  ("df_in_ols",  "y_pred_ols"),
    "PLS":  ("df_in_pls",  "y_pred_pls"),
    "PCR":  ("df_in_pcr",  "y_pred_pcr"),
    "Enet": ("df_in_en", "y_pred_en"),
    "RF":   ("df_in_rf",   "y_pred_rf"),
    "GBRT": ("df_in_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_in_xgb",  "y_pred_xgb")
}

r2_results_in = {}
for model, (df_name, pred_col) in models.items():
    df = globals()[df_name]   # récupère la variable df_in_xxx par son nom
    r2_val = r2(df["y_true"], df[pred_col])
    r2_results_in[model] = r2_val

print(r2_results_in)

models_oos = {
    "OLS":  ("df_oos_ols",  "y_pred_ols"),
    "PLS":  ("df_oos_pls",  "y_pred_pls"),
    "PCR":  ("df_oos_pcr",  "y_pred_pcr"),
    "Enet": ("df_oos_en", "y_pred_en"),
    "RF":   ("df_oos_rf",   "y_pred_rf"),
    "GBRT": ("df_oos_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_oos_xgb",  "y_pred_xgb")
}

r2_results_oos = {}

for model, (df_name, pred_col) in models_oos.items():
    df = globals()[df_name]   # car tu stockes des morceaux dans une liste
    r2_val = r2(df["y_true"], df[pred_col])
    r2_results_oos[model] = float(r2_val)

print(r2_results_oos)

# IN-SAMPLE — R² brut de HA
tmp_in = (
    df_in_ols[["Date","Ticker","y_true"]]
    .merge(df_in_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
r2_ha_in = float(r2(tmp_in["y_true"], tmp_in["ha_return"]))
print("R²_in (HA) :", r2_ha_in)

# OUT-OF-SAMPLE — R² brut de HA
tmp_oos = (
    df_oos_ols[["Date","Ticker","y_true"]]
    .merge(df_oos_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
r2_ha_oos = float(r2(tmp_oos["y_true"], tmp_oos["ha_return"]))
print("R²_oos (HA):", r2_ha_oos)

# Option : ajoute-les à tes dicts de résultats pour les tables
r2_results_in["HA"]  = r2_ha_in
r2_results_oos["HA"] = r2_ha_oos

In [ ]:
#R² - Benchmark HA - Xiu et Liu 

#R2 in
df_in_ha_bench = df_in_ha[["Date", "Ticker", "ha_return"]]

r2_vs_ha_in = {}

for model, (df_name, pred_col) in models.items():
    df = globals()[df_name]
    tmp = df.merge(df_in_ha_bench, on=["Date", "Ticker"], how="inner") #permet de comparer les lignes ayant les mêmes dates et tickers
    r2_vs_ha_in[model] = float(
        r2_vs_benchmark(tmp["y_true"].to_numpy(),
                        tmp[pred_col].to_numpy(),
                        tmp["ha_return"].to_numpy())
    )

print(r2_vs_ha_in)

r2_vs_ha_oos = {}
df_oos_ha_bench = df_oos_ha[["Date", "Ticker", "ha_return"]]

for model, (df_name, pred_col) in models_oos.items():
    df = globals()[df_name]
    # Alignement sur Date + Ticker
    tmp = df.merge(df_oos_ha_bench, on=["Date", "Ticker"], how="inner")
    # R² vs benchmark HA
    r2_vs_ha_oos[model] = float(
        r2_vs_benchmark(tmp["y_true"].to_numpy(),
                        tmp[pred_col].to_numpy(),
                        tmp["ha_return"].to_numpy())
    )

print(r2_vs_ha_oos)

In [ ]:
#Success ratio - Xiu and Liu 
success_in = {}
for model, (df_name, pred_col) in {
    "OLS":  ("df_in_ols",  "y_pred_ols"),
    "PLS":  ("df_in_pls",  "y_pred_pls"),
    "PCR":  ("df_in_pcr",  "y_pred_pcr"),
    "Enet": ("df_in_en",   "y_pred_en"),
    "RF":   ("df_in_rf",   "y_pred_rf"),
    "GBRT": ("df_in_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_in_xgb",  "y_pred_xgb")
}.items():
    df = globals()[df_name]
    success_in[model] = float(success_ratio(df["y_true"], df[pred_col]))

print("Success ratio IN:", success_in)

success_oos = {}
for model, (df_name, pred_col) in {
    "OLS":  ("df_oos_ols",  "y_pred_ols"),
    "PLS":  ("df_oos_pls",  "y_pred_pls"),
    "PCR":  ("df_oos_pcr",  "y_pred_pcr"),
    "Enet": ("df_oos_en",   "y_pred_en"),
    "RF":   ("df_oos_rf",   "y_pred_rf"),
    "GBRT": ("df_oos_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_oos_xgb",  "y_pred_xgb")
}.items():
    df = globals()[df_name]
    success_oos[model] = float(success_ratio(df["y_true"], df[pred_col]))

print("Success ratio OOS:", success_oos)

# R2 in sample HA
tmp_in = (
    df_in_ols[["Date","Ticker","y_true"]]
    .merge(df_in_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
sr_ha_in = float(success_ratio(tmp_in["y_true"], tmp_in["ha_return"]))

# R2 oos HA
tmp_oos = (
    df_oos_ols[["Date","Ticker","y_true"]]
    .merge(df_oos_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
sr_ha_oos = float(success_ratio(tmp_oos["y_true"], tmp_oos["ha_return"]))

print({"HA_in": sr_ha_in, "HA_oos": sr_ha_oos})

In [ ]:
#Création des tables latex 
models_list = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB", "HA"]

#Table 1 : R²-in sample brut, r2-IS vs HA, Success ratio in sample 
rows = []
for model in models_list:
    if model == "HA":
        r2_in = r2_ha_in          
        r2_vs_ha = 0.0           
        sr_in = sr_ha_in
    else:
        r2_in = r2_results_in[model]
        r2_vs_ha = r2_vs_ha_in[model]
        sr_in = success_in[model]

    rows.append({
        "Model": model,
        "In-sample $R^2$": r2_in,
        "In-sample $R^2$ vs HA": r2_vs_ha,
        "In-sample Success Ratio": sr_in
    })

df_table1 = pd.DataFrame(rows)

# Mise en forme
for col in ["In-sample $R^2$", "In-sample $R^2$ vs HA", "In-sample Success Ratio"]:
    df_table1[col] = (df_table1[col].astype(float) * 100).apply(
        lambda x: f"{x:.2f}" if not np.isnan(x) else ""
    )

latex_table1 = df_table1.to_latex(index=False, escape=False, column_format="lccc")
print(latex_table1)

#Table 2 : R2 in sample / R2 out of sample (Gu et al. comparaison)
models_list_noha = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB"]

rows = []
for model in models_list_noha:
    r2_in  = r2_results_in[model]
    r2_oos = r2_results_oos[model]
    rows.append({
        "Model": model,
        "In-sample $R^2$": r2_in,
        "Out-of-sample $R^2$": r2_oos
    })

df_table2 = pd.DataFrame(rows)

# Mise en forme : *100 et arrondi à 2 décimales
for col in ["In-sample $R^2$", "Out-of-sample $R^2$"]:
    df_table2[col] = (df_table2[col].astype(float) * 100).apply(lambda x: f"{x:.2f}")

latex_table2 = df_table2.to_latex(index=False, escape=False, column_format="lcc")
print(latex_table2)

# Table 3 : R² vs HA (OOS) + Success Ratio OOS, avec HA
models_all = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB", "HA"]

rows = []
for model in models_all:
    if model == "HA":
        r2_vs_ha = 0.0              # par définition, HA vs HA = 0
        sr_oos   = sr_ha_oos        # success ratio de HA
    else:
        r2_vs_ha = float(r2_vs_ha_oos[model])
        sr_oos   = float(success_oos[model])

    rows.append({
        "Model": model,
        "Out-of-sample $R^2$ vs HA": r2_vs_ha,
        "Out-of-sample Success Ratio": sr_oos
    })

df_table3 = pd.DataFrame(rows)

# Mise en forme : *100 et arrondi à 2 décimales
for col in ["Out-of-sample $R^2$ vs HA", "Out-of-sample Success Ratio"]:
    df_table3[col] = (df_table3[col].astype(float) * 100).apply(lambda x: f"{x:.2f}")

latex_table3 = df_table3.to_latex(index=False, escape=False, column_format="lcc")
print(latex_table3)


Diebold

In [ ]:
#Vérif : y true bons
models = {
    "OLS": df_oos_ols,
    "PLS": df_oos_pls,
    "PCR": df_oos_pcr,
    "Enet": df_oos_en,
    "RF": df_oos_rf,
    "GBRT": df_oos_gbrt,
    "XGB": df_oos_xgb,
}

# choisir une référence, par ex OLS
ref_name, ref_df = "OLS", models["OLS"]

for name, df in models.items():
    if name == ref_name:
        continue
    check = ref_df.merge(df, on=["Date","Ticker"], suffixes=("_ref", f"_{name}"))
    mismatches = (check["y_true_ref"] != check[f"y_true_{name}"]).sum()
    print(f"Différences de y_true entre {ref_name} et {name} :", mismatches)

In [ ]:
#Créations tableaux prédictions
df_oos = df_oos_ols.copy()[["Date","Ticker","y_true","y_pred_ols"]]

models = {
    "PLS": (df_oos_pls, "y_pred_pls"),
    "PCR": (df_oos_pcr, "y_pred_pcr"),
    "EN": (df_oos_en, "y_pred_en"),
    "RF": (df_oos_rf, "y_pred_rf"),
    "GBRT": (df_oos_gbrt, "y_pred_gbrt"),
    "XGB": (df_oos_xgb, "y_pred_xgb"),
    "HA": (df_oos_ha, "ha_return")
}

for name, (df_model, col_pred) in models.items():
    df_oos = df_oos.merge(
        df_model[["Date","Ticker",col_pred]].rename(columns={col_pred:name}),
        on=["Date","Ticker"],
        how="left"
    )

df_oos.rename(columns={"y_pred_ols": "OLS"}, inplace=True)

print(df_oos.head())

In [ ]:
#DIABOLD TEST 

df_diebold = df_oos[["y_true","OLS","PLS","PCR","EN","RF","GBRT","XGB","HA"]].copy()

results = []

models = [c for c in df_diebold.columns if c not in ["y_true","Date","Ticker"]]

for i in range(len(models)):
    for j in range(i+1, len(models)):
        model1 = models[i]
        model2 = models[j]

        #calcule mse du modèle 1 et 2 
        e1 = (df_diebold['y_true'] - df_diebold[model1])**2
        e2 = (df_diebold['y_true'] - df_diebold[model2])**2
        d = e1 - e2
        d = d.dropna()

        X = np.ones(len(d))  # régression constante
        model = sm.OLS(d, X).fit()
        cov = cov_hac(model, nlags=1)  
        se = np.sqrt(cov[0][0])
        dm_stat = d.mean() / se
        p_value = 2 * (1 - t.cdf(abs(dm_stat), df=len(d) - 1))

        results.append({
            'Model 1': model1,
            'Model 2': model2,
            'DM Stat': dm_stat,
            'P-Value': p_value,
            'Best Model': model2 if dm_stat > 0 else model1
        })


dm_df = pd.DataFrame(results)
dm_df_sorted = dm_df.sort_values(by="P-Value", ascending=True)
#print(dm_df_sorted)
print(dm_df)